# Baseline: MASSIVE en-US pre-processing

Turns the MASSIVE en-US file into encoded data for B1, M1 and M4, following the team's [data loading guide](https://claude.ai/artifact/P7ZiwdxinFuBRcYL9Qtn3K).
Run top to bottom. After step 7 the notebook holds the encoded splits, every vocabulary, and two 5,304 × 300 embedding matrices.
Padding and masks come later, when batching.

Each step lists the numbers the guide expects. They were measured on a version 1.0 copy of the file.

In [1]:
import glob, json, os, random, re, tarfile, urllib.request, zipfile

import numpy as np

DATA_DIR = "data"   # all downloads go here; git-ignored
os.makedirs(DATA_DIR, exist_ok=True)

## Step 1: Download the data

Only one file from the official release is used: `en-US.jsonl`, with the split marked on each row.
To use version 1.0 instead, change the URL's ending to `1.0.tar.gz` and re-run the notebook.

**Expected:** `en-US.jsonl` found on disk.

In [2]:
url = "https://amazon-massive-nlu-dataset.s3.amazonaws.com/amazon-massive-dataset-1.1.tar.gz"
archive = os.path.join(DATA_DIR, os.path.basename(url))
if not os.path.exists(archive):
    urllib.request.urlretrieve(url, archive)
with tarfile.open(archive) as tar:
    tar.extractall(os.path.join(DATA_DIR, "massive"), filter="data")
DATA_PATH = glob.glob(os.path.join(DATA_DIR, "massive", "**", "en-US.jsonl"), recursive=True)[0]
print(DATA_PATH)

data\massive\1.1\data\en-US.jsonl


## Step 2: Read the file and split it

Train fits the models, dev tunes them and picks checkpoints, and test is used once at the end.

**Expected:** 11,514 / 2,033 / 2,974 rows.

In [3]:
def load_massive(path):
    splits = {"train": [], "dev": [], "test": []}
    with open(path, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                row = json.loads(line)
                splits[row["partition"]].append(row)
    return splits

splits = load_massive(DATA_PATH)
print({k: len(v) for k, v in splits.items()})

{'train': 11514, 'dev': 2033, 'test': 2974}


## Step 3: Convert annotations into words and BIO tags

The models need one tag per word, so each `annot_utt` becomes a word list plus a matching BIO tag list.
`to_examples` also checks that the words match `utt.split()`, so input words and tags always line up.

**Expected:** all checks pass; 11,344 slot spans in train; about 32% of utterances have no slots.

In [4]:
SLOT_RE = re.compile(r"\[(\S+) : ([^\]]+)\]")

def annot_to_bio(annot):
    tokens, tags, pos = [], [], 0
    for m in SLOT_RE.finditer(annot):
        for w in annot[pos:m.start()].split():        # words before the slot
            tokens.append(w); tags.append("O")
        words = m.group(2).split()                     # words inside the slot
        tokens += words
        tags += [f"B-{m.group(1)}"] + [f"I-{m.group(1)}"] * (len(words) - 1)
        pos = m.end()
    for w in annot[pos:].split():                      # words after the last slot
        tokens.append(w); tags.append("O")
    return tokens, tags

def to_examples(rows):
    out = []
    for row in rows:
        tokens, tags = annot_to_bio(row["annot_utt"])
        assert tokens == row["utt"].split(), row["id"]
        out.append({"id": row["id"], "tokens": tokens, "tags": tags,
                    "intent": row["intent"], "annot_utt": row["annot_utt"]})
    return out

examples = {split: to_examples(rows) for split, rows in splits.items()}
spans = sum(t.startswith("B-") for e in examples["train"] for t in e["tags"])
no_slot = sum(all(t == "O" for t in e["tags"]) for e in examples["train"])
print(f"train: {spans} slot spans, {100 * no_slot / len(examples['train']):.1f}% of utterances have no slots")

train: 11344 slot spans, 32.6% of utterances have no slots


### Parsing sanity check

The automatic checks must all pass; the five printed examples are for a visual check that words and tags line up.

**Expected:**

```
train: 11514 utterances, 3755 with no slots, 0 invalid I- tags
dev: 2033 utterances, 646 with no slots, 0 invalid I- tags
test: 2974 utterances, 994 with no slots, 0 invalid I- tags
```

In [5]:
def bio_to_annot(tokens, tags):
    """Rebuild 'wake me up at [time : five am]' from words and BIO tags."""
    out, i = [], 0
    while i < len(tokens):
        if tags[i] == "O":
            out.append(tokens[i]); i += 1; continue
        label, span = tags[i][2:], [tokens[i]]; i += 1
        while i < len(tokens) and tags[i] == f"I-{label}":
            span.append(tokens[i]); i += 1
        out.append(f"[{label} : {' '.join(span)}]")
    return " ".join(out)

# Automatic checks: every one must pass
for split, exs in examples.items():
    assert all(len(e["tokens"]) == len(e["tags"]) for e in exs), split
    assert all(bio_to_annot(e["tokens"], e["tags"]) == " ".join(e["annot_utt"].split()) for e in exs), split
    bad_i = sum(t.startswith("I-") and p[2:] != t[2:]
                for e in exs for p, t in zip(["O"] + e["tags"], e["tags"]))
    no_slot = sum(all(t == "O" for t in e["tags"]) for e in exs)
    print(f"{split}: {len(exs)} utterances, {no_slot} with no slots, {bad_i} invalid I- tags")

# Five random examples, words lined up with their tags
random.seed(42)
for i, ex in enumerate(random.sample(examples["train"], 5), 1):
    w = [max(len(a), len(b)) for a, b in zip(ex["tokens"], ex["tags"])]
    print(f"\n[{i}] id={ex['id']}  intent={ex['intent']}")
    print("    utt      :", " ".join(ex["tokens"]))
    print("    annot_utt:", ex["annot_utt"])
    print("    words    :", "  ".join(a.ljust(n) for a, n in zip(ex["tokens"], w)))
    print("    tags     :", "  ".join(b.ljust(n) for b, n in zip(ex["tags"], w)))

train: 11514 utterances, 3755 with no slots, 0 invalid I- tags
dev: 2033 utterances, 646 with no slots, 0 invalid I- tags
test: 2974 utterances, 994 with no slots, 0 invalid I- tags

[1] id=15621  intent=social_post
    utt      : complaint
    annot_utt: complaint
    words    : complaint
    tags     : O        

[2] id=2680  intent=weather_query
    utt      : is it going to warm up soon
    annot_utt: is it going to [weather_descriptor : warm up] [time : soon]
    words    : is  it  going  to  warm                  up                    soon  
    tags     : O   O   O      O   B-weather_descriptor  I-weather_descriptor  B-time

[3] id=636  intent=general_joke
    utt      : what is a funny joke about car
    annot_utt: what is a [joke_type : funny] joke about [joke_type : car]
    words    : what  is  a  funny        joke  about  car        
    tags     : O     O   O  B-joke_type  O     O      B-joke_type

[4] id=6712  intent=news_query
    utt      : ethics
    annot_utt: ethics


## Steps 4 and 5: Build vocabularies and label maps

Every vocabulary is built from **train only**, so dev and test stay unseen. Each one is sorted, so ids are reproducible.
The tag map lists every `B-` and `I-` tag, including `I-` tags train never uses, because those still need an id.

**Expected:**
- Step 4: 60 intents; 111 tags (`O` + B-/I- for 55 slot types), 107 of them seen in train.
- Step 5: 5,304 words (5,302 + 2); 34 characters (32 + 2); `<PAD>` = 0 and `<UNK>` = 1.

In [6]:
def build_vocabs(train):
    intents = sorted({e["intent"] for e in train})
    slot_types = sorted({t[2:] for e in train for t in e["tags"] if t != "O"})
    tags = ["O"] + [f"{prefix}-{slot}" for slot in slot_types for prefix in "BI"]
    words = sorted({w for e in train for w in e["tokens"]})
    chars = sorted({c for w in words for c in w})
    specials = ["<PAD>", "<UNK>"]
    return {"word": {w: i for i, w in enumerate(specials + words)},
            "char": {c: i for i, c in enumerate(specials + chars)},
            "intent": {name: i for i, name in enumerate(intents)},
            "tag": {t: i for i, t in enumerate(tags)}}

vocabs = build_vocabs(examples["train"])
word2id, char2id, intent2id, tag2id = vocabs["word"], vocabs["char"], vocabs["intent"], vocabs["tag"]

seen_tags = {t for e in examples["train"] for t in e["tags"]}
print(f"step 4: {len(intent2id)} intents; {len(tag2id)} tags, {len(seen_tags)} seen in train")
print(f"step 5: {len(word2id)} words; {len(char2id)} characters; <PAD> = {word2id['<PAD>']}, <UNK> = {word2id['<UNK>']}")

step 4: 60 intents; 111 tags, 107 seen in train
step 5: 5304 words; 34 characters; <PAD> = 0, <UNK> = 1


## Step 6: Build the GloVe and fastText matrices

Both matrices follow `word2id`'s order, so row *i* is the vector for word id *i*. `build_matrix` scans each file once and keeps only the vocabulary's rows.
The first run downloads about 1.45 GB of zips and leaves about 3.3 GB of extracted vectors in `data/`. The zips are deleted after extraction.

**Expected:** each matrix 5,304 × 300 (1,591,200 numbers), `<PAD>` row all zeros, and:

```
GloVe: matrix (5304, 300), missing 184 of 5302 words (3.47%), e.g. ['<empty>', "altucher's", 'apahelp', 'apig', 'applebees', 'badpho']
fastText: matrix (5304, 300), missing 140 of 5302 words (2.64%), e.g. ['<empty>', 'apahelp', "aren't", 'badpho', 'bbcnews', 'beanos']
missing from both: 126
```

The guide measured these counts on the gensim-data copies of the same two files.

In [7]:
downloads = {
    "glove.6B.300d.txt": "https://huggingface.co/stanfordnlp/glove/resolve/main/glove.6B.zip",
    "wiki-news-300d-1M-subword.vec": "https://dl.fbaipublicfiles.com/fasttext/vectors-english/wiki-news-300d-1M-subword.vec.zip",
}
for filename, url in downloads.items():
    if not os.path.exists(os.path.join(DATA_DIR, filename)):
        zip_path = os.path.join(DATA_DIR, os.path.basename(url))
        urllib.request.urlretrieve(url, zip_path)
        with zipfile.ZipFile(zip_path) as z:
            z.extract(filename, DATA_DIR)
        os.remove(zip_path)

In [8]:
def split_apostrophe(word):
    """amazon's -> ('amazon', "'s"); aren't -> ('are', "n't")"""
    i = len(word) - 3 if word.endswith("n't") else word.index("'")
    return word[:i], word[i:]

def find(word, table):
    """Exact match, then Capitalised and UPPER forms (fastText is case-sensitive)."""
    for w in (word, word.capitalize(), word.upper()):
        if w in table:
            return table[w]
    return None

def lookup(word, table):
    vec = find(word, table)
    if vec is None and "'" in word:          # average the pieces GloVe/fastText store separately
        pieces = [find(p, table) for p in split_apostrophe(word)]
        if all(p is not None for p in pieces):
            vec = np.mean(pieces, axis=0)
    return vec

def build_matrix(path, word2id, dim=300, seed=0):
    """len(word2id) x dim matrix in word2id's order. <PAD> = zeros, <UNK> and missing words = random.
    Returns (matrix, list of vocabulary words that were not found)."""
    wanted = set(word2id)
    for w in word2id:
        if "'" in w:
            wanted.update(split_apostrophe(w))
    table = {}
    with open(path, encoding="utf-8") as f:
        for line in f:
            parts = line.rstrip().split(" ")          # rstrip(): fastText lines end with a space
            if len(parts) == dim + 1 and parts[0].lower() in wanted:   # also skips the header line
                table[parts[0]] = np.asarray(parts[1:], dtype=np.float32)
    rng = np.random.default_rng(seed)
    scale = np.std(np.stack(list(table.values())))   # random vectors on the same scale as real ones
    matrix = np.zeros((len(word2id), dim), dtype=np.float32)
    missing = []
    for w, idx in word2id.items():
        if w == "<PAD>":
            continue
        vec = None if w == "<UNK>" else lookup(w, table)
        if vec is None:
            vec = rng.normal(0.0, scale, dim)
            if w != "<UNK>":
                missing.append(w)
        matrix[idx] = vec
    return matrix, missing

glove_matrix, glove_missing = build_matrix(os.path.join(DATA_DIR, "glove.6B.300d.txt"), word2id)
fasttext_matrix, fasttext_missing = build_matrix(os.path.join(DATA_DIR, "wiki-news-300d-1M-subword.vec"), word2id)
n = len(word2id) - 2
for name, m, miss in [("GloVe", glove_matrix, glove_missing), ("fastText", fasttext_matrix, fasttext_missing)]:
    print(f"{name}: matrix {m.shape}, missing {len(miss)} of {n} words ({100 * len(miss) / n:.2f}%), e.g. {miss[:6]}")
    assert not m[word2id["<PAD>"]].any()
print("missing from both:", len(set(glove_missing) & set(fasttext_missing)))

GloVe: matrix (5304, 300), missing 184 of 5302 words (3.47%), e.g. ['<empty>', "altucher's", 'apahelp', 'apig', 'applebees', 'badpho']
fastText: matrix (5304, 300), missing 140 of 5302 words (2.64%), e.g. ['<empty>', 'apahelp', "aren't", 'badpho', 'bbcnews', 'beanos']
missing from both: 126


## Step 7: Encode every sentence

Unseen words and characters in dev and test map to `<UNK>` (id 1). Lists stay variable-length; padding comes later, when batching.

**Expected:** same counts as step 2, with one tag id per word. The one-word utterance "complaint" (id 15621) encodes to:

```python
{'word_ids': [974], 'char_ids': [[10, 22, 20, 23, 19, 8, 16, 21, 27]], 'tag_ids': [0], 'intent_id': 51}
```

In [9]:
def encode(ex, vocabs):
    word2id, char2id = vocabs["word"], vocabs["char"]
    return {"word_ids": [word2id.get(w, word2id["<UNK>"]) for w in ex["tokens"]],
            "char_ids": [[char2id.get(c, char2id["<UNK>"]) for c in w] for w in ex["tokens"]],
            "tag_ids": [vocabs["tag"][t] for t in ex["tags"]],
            "intent_id": vocabs["intent"][ex["intent"]]}

encoded = {split: [encode(ex, vocabs) for ex in exs] for split, exs in examples.items()}

for split, xs in encoded.items():
    assert all(len(x["word_ids"]) == len(x["char_ids"]) == len(x["tag_ids"]) for x in xs), split
print({k: len(v) for k, v in encoded.items()})
print(next(x for e, x in zip(examples["train"], encoded["train"]) if e["id"] == "15621"))

{'train': 11514, 'dev': 2033, 'test': 2974}
{'word_ids': [974], 'char_ids': [[10, 22, 20, 23, 19, 8, 16, 21, 27]], 'tag_ids': [0], 'intent_id': 51}


## Sanity checks

Run this after pre-processing and before training anything. If a number differs, check the dataset version and the file path first.

| Check | Expected |
| --- | --- |
| Utterances (train / dev / test) | 11,514 / 2,033 / 2,974 |
| Vocab sizes (word / char / intent / tag) | 5,304 / 34 / 60 / 111 |
| Embedding parameters at 300d | 1,591,200 |
| Words per train utterance | 6.92 on average, 35 at most |
| Dev / test utterances with an unseen word | 17.90% / 18.93% |

In [10]:
print("utterances:", {k: len(v) for k, v in examples.items()})
print("vocab sizes:", {"word": len(word2id), "char": len(char2id), "intent": len(intent2id), "tag": len(tag2id)})
print("numbers per 300d embedding matrix:", len(word2id) * 300)
lens = [len(e["tokens"]) for e in examples["train"]]
print(f"train words per utterance: mean {np.mean(lens):.2f}, max {max(lens)}")
for split in ("dev", "test"):
    unseen = sum(any(w not in word2id for w in e["tokens"]) for e in examples[split])
    print(f"{split} utterances with an unseen word: {100 * unseen / len(examples[split]):.2f}%")

utterances: {'train': 11514, 'dev': 2033, 'test': 2974}
vocab sizes: {'word': 5304, 'char': 34, 'intent': 60, 'tag': 111}
numbers per 300d embedding matrix: 1591200
train words per utterance: mean 6.92, max 35
dev utterances with an unseen word: 17.90%
test utterances with an unseen word: 18.93%
